In [23]:
import pandas as pd

df = pd.read_csv('../공유/final_cleaned_airbnb.csv')

In [24]:
print(df['room_capacity'].describe())

count    19861.000000
mean         4.073586
std          2.135081
min          0.000000
25%          3.000000
50%          3.000000
75%          5.000000
max         71.500000
Name: room_capacity, dtype: float64


##### property_type 전처리

In [25]:
#기본정보 확인
print(df['property_type'].describe())

count                  19861
unique                    12
top       Entire rental unit
freq                    8661
Name: property_type, dtype: object


In [26]:
#결측치 개수 확인
nan_count = df['property_type'].isna().sum()
print(f"결측치 개수: {nan_count}개")

결측치 개수: 0개


In [27]:
#숙소 유형별로 얼마나 있는지 확인
print("숙소 유형별 빈도수 (Top 10)")
property_counts = df['property_type'].value_counts()
print(property_counts.head(10))

숙소 유형별 빈도수 (Top 10)
property_type
Entire rental unit             8661
Private room in rental unit    5189
Private room in home           1886
Entire home                     857
Room in hotel                   814
Entire condo                    626
Private room in townhouse       622
Private room in condo           341
Entire townhouse                272
Entire loft                     234
Name: count, dtype: int64


In [28]:
#결측치가 없으니 통계 과정에서 노이즈가 될 수 있는 데이터가 뭐가 있을까 고민해봄
#llm에게 물어보니까 -> 데이터가 너무 작으면 분석에 방해가 될 수 있다고 함.
#그래서 전체 데이터의 1% 미만인 데이터들을 뽑아봄
prop_counts = df['property_type'].value_counts()
prop_pct = df['property_type'].value_counts(normalize=True) * 100

# 1% 미만 비중을 차지하는 데이터
rare = pd.DataFrame({'Count': prop_counts, 'Percentage': prop_pct})
rare_types = rare[rare['Percentage'] < 1.0]

#편의상 1%미만의 데이터를 희귀 숙소라고 지칭
print("\n희귀 숙소 분석")
print(f"종류 수: {len(rare_types)}개")
print(f"전체 데이터에서 차지하는 비중: {rare_types['Percentage'].sum():.2f}%")
print("가장 희소한 숙소:", rare_types.tail(5).index.tolist())
#목장, 탑, 돔 등 -> 숙소보다는 체험?같은걸 하기 위해 방문하는 곳인듯 함


희귀 숙소 분석
종류 수: 1개
전체 데이터에서 차지하는 비중: 0.74%
가장 희소한 숙소: ['Entire guest suite']


In [29]:
#이러한 데이터가 있다는 것을 생각하고, 소희.찬희님이 해주신 것처럼 희귀숙소를 other로 묶어서 분류하면 될 듯 

##### room_type 전처리

In [30]:
print(df['room_type'].value_counts(dropna=False))
#데이터 타입 : 범주형

room_type
Entire home/apt    10797
Private room        8704
Hotel room           360
Name: count, dtype: int64


In [31]:
#비어있는(NaN) 행이 있는지 확인
room_nan = df['room_type'].isna().sum()
print(f"\nroom_type 결측치: {room_nan}개")


room_type 결측치: 0개


##### propety_type과 room_type 비교전처리

In [32]:
#room_type은 entire인데 property_type이 'Private room'인 경우가 있을지?
#'room_type'이 Entire home/apt 인 데이터만 따로 추출
entire_homes = df[df['room_type'] == 'Entire home/apt']

#property_type에 'Private'이라는 글자가 들어간 행이 있는지 확인
#llm에게 물어봄 -> 대소문자 구분 없이 찾기 위해서 case=False 사용
mismatch = entire_homes[entire_homes['property_type'].str.contains('Private', case=False, na=False)]

print(f"전체 대여(Entire) 중 'Private' 문구가 포함된 데이터: {len(mismatch)}건")
#room_type이 entire인데 property_type이 'Private'인 경우는 없음 -> 데이터가 일관적임

전체 대여(Entire) 중 'Private' 문구가 포함된 데이터: 0건


##### accomodates 전처리

In [33]:
print(df['accommodates'].describe())
#데이터 타입 : 실수형

count    19861.00000
mean         2.81295
std          1.97293
min          1.00000
25%          2.00000
50%          2.00000
75%          4.00000
max         16.00000
Name: accommodates, dtype: float64


In [34]:
#결측치 및 0인 데이터 확인
nan = df['accommodates'].isna().sum()
zero = (df['accommodates'] == 0).sum()
print(f"\n결측치 개수: {nan}개")
print(f"0인 데이터: {zero}개")


결측치 개수: 0개
0인 데이터: 0개


In [35]:
#이상치 확인 (너무 큰 숫자 뽑아내기)
quantile_99 = df['accommodates'].quantile(0.99)
over_10 = df[df['accommodates'] > 10]

print(f"\n이상치 분석")
print(f"상위 1% 기준: {quantile_99}명")
print(f"10명 초과 수용 숙소 개수: {len(over_10)}개")
#보통 뉴욕 아파트에서 10명 이상 수용은 드문 경우라고 함


이상치 분석
상위 1% 기준: 10.0명
10명 초과 수용 숙소 개수: 189개


뉴욕 기준의 법적/안전적 근거
뉴욕 시는 소방 안전 및 주거법상 한 공간에 머물 수 있는 인원을 엄격하게 제한함
방 하나에 8명 이상이 있는 것은 뉴욕 주거법상 불법일 확률이 매우 높음

즉, 데이터에 10명, 12명이라고 적혀 있다면 호스트가 검색 노출을 위해 수용 인원으로 거짓말을 쳤거나, 정말 특이한 형태의 불법 숙소일 수 있수도 있음

In [36]:
#room_type과의 논리적 모순 체크
#'Shared room'인데 수용 인원이 10명이다? (도미토리(게하)일 수도 있지만 체크가 필요하다고 생각했음)
shared_large = df[(df['room_type'] == 'Shared room') & (df['accommodates'] > 8)]
print(f"\nShared인데 8명 초과인 숙소: {len(shared_large)}개")
if not shared_large.empty:
    print(shared_large[['property_type', 'accommodates']].head())


Shared인데 8명 초과인 숙소: 0개


In [37]:
# Shared room만 필터링해서 수용 인원의 분포 확인
shared_rooms = df[df['room_type'] == 'Shared room']

print("Shared room 수용 인원 분위수")
print(shared_rooms['accommodates'].quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))
#shared room 데이터는 깔끔해서 그대로 가져가도 될듯함

Shared room 수용 인원 분위수
0.25   NaN
0.50   NaN
0.75   NaN
0.90   NaN
0.95   NaN
0.99   NaN
Name: accommodates, dtype: float64


##### bathrooms를 정수형으로 바꾸고, amenities 에서 새로운 파생 컬럼 만들어보기

In [38]:
print(df['bathrooms'].describe())

count    19861.000000
mean         1.190348
std          0.552352
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         15.500000
Name: bathrooms, dtype: float64


In [39]:
df['bathrooms'] = df['bathrooms'].fillna(0).astype(float)
# 결측치를 채워준 이유 -> 숙소에 욕실이 없는 경우도 있을 수 있기 때문에, 결측치를 0으로 채워주는 것이 논리적으로 맞다고 판단함

In [40]:
df['bathrooms'] = df['bathrooms'].fillna(0).astype(float)
df['bedrooms'] = df['bedrooms'].fillna(0).astype(float)
df['beds'] = df['beds'].fillna(0).astype(float)

#파생 컬럼 생성: room_capacity
#숙소의 규모를 결정하는 요소 합산
df['room_capacity'] = df['bathrooms'] + df['bedrooms'] + df['beds']

cols = ['bathrooms', 'bedrooms', 'beds', 'room_capacity']
print(df[cols].sort_values(by='room_capacity', ascending=False).head())

print(df['room_capacity'].describe())

       bathrooms  bedrooms  beds  room_capacity
698         15.5      14.0  42.0           71.5
4276         4.0       9.0  13.0           26.0
12420        7.5       9.0   9.0           25.5
3498         4.0       7.0  14.0           25.0
12025        4.0       9.0  12.0           25.0
count    19861.000000
mean         4.073586
std          2.135081
min          0.000000
25%          3.000000
50%          3.000000
75%          5.000000
max         71.500000
Name: room_capacity, dtype: float64


In [43]:
# 상위 5개 숙소의 세부 정보 확인 (동네, 가격 등)
top_5 = df.sort_values(by='room_capacity', ascending=False).head(5)

# 보고 싶은 정보들만 쏙쏙 골라서 출력
cols_to_see = ['id','name','neighbourhood_group_cleansed', 'room_type', 'room_capacity', 'price']
display(top_5[cols])

,bathrooms,bedrooms,beds,room_capacity
698,15.5,14.0,42.0,71.5
4276,4.0,9.0,13.0,26.0
12420,7.5,9.0,9.0,25.5
3498,4.0,7.0,14.0,25.0
12025,4.0,9.0,12.0,25.0


In [ ]:
import ast #llm에게 물어봄 -> 문자열로 된 리스트를 실제 리스트로 변환해주는 라이브러리

#amenities 한 행에 대한 값 = x
def get_len_safe(x): #try-except : 오류가 날 수 있는 상황에서 안전하게 처리하기 위한 구문
    try: #try : 오류가 날 수 있는 상황에서 시도할 코드 블록
        if pd.isna(x): return 0 
        # ast.literal_eval: 텍스트로 된 리스트를 실제 리스트 객체로 변환
        # len(...): 변환된 리스트 안의 요소 개수를 카운트
        return len(ast.literal_eval(x)) #ast.literal_eval(x):['beds','bedrooms','bathrooms']
    except:
        # 데이터가 리스트 형식이 아니면 그냥 0으로 반환 
        return 0
    #ast.literal_eval : 문자열(String) 형태로 되어 있는 데이터를 파이썬의 실제 데이터 타입(리스트, 딕셔너리, 튜플 등)으로 안전하게 변환해주는 함수

#전체 데이터프레임의 'amenities' 컬럼에 위 함수를 적용하여 개수 계산
#결과는 각 행의 개수만 담긴 Series 형태가 됨
lengths = df['amenities'].apply(get_len_safe)

#개수가 많은 순서대로 내림차순 정렬 후, 상위 5개의 인덱스(번호표)만 추출
top_indices = lengths.sort_values(ascending=False).index[:5]

#추출한 인덱스를 이용해 원본 데이터프레임에서 해당 숙소들의 전체 정보를 가져옴
#.copy()를 써서 원본 데이터를 보호하고 독립된 복사본 생성
top_amenities = df.loc[top_indices].copy()

#'amenities_len'이라는 새 컬럼을 만들고, 아까 계산한 개수 값을 인덱스에 맞춰 정확히 입력
# 이 작업을 거쳐야 표에 NaN이 생기지 않고 숫자가 제대로 들어감
#NaN!!!!! 방지를 위해 .loc로 직접 값 넣어주기
top_amenities['amenities_len'] = lengths[top_indices]

display_cols = ['id', 'name', 'amenities_len', 'price', 'neighbourhood_group_cleansed']
display(top_amenities[display_cols])

,id,name,amenities_len,price,neighbourhood_group_cleansed
7806,51522141,Study zone near Columbia •••,95,100.0,Manhattan
3442,24312253,Study zone near Columbia ••,92,98.0,Manhattan
3443,24312535,Study zone near Columbia •,91,157.0,Manhattan
6134,44804027,"Stunning Two-Story 3 Bedroom, 2 Bathroom Flat",89,482.0,Manhattan
4448,33521985,Red door oasis 3 bedroom * near jfk /time square,88,360.0,Queens


In [ ]:
df.to_csv("airbnb.csv", index=False)